# Online Retail II: Data Cleaning
**Goal:** Profile the raw data, document every quality issue, and produce a clean dataset for analysis.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/online_retail_II.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Shape:", df.shape)
df.head()

Shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
df.info()
print(df.isnull().sum())
print((df.isnull().mean() * 100).round(2))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Pr

In [4]:
before = {
    "Total rows": len(df),
    "Missing Customer ID": df["Customer ID"].isnull().sum(),
    "Missing Description": df["Description"].isnull().sum(),
    "Exact duplicate rows": df.duplicated().sum(),
    "Quantity <= 0": (df["Quantity"] <= 0).sum(),
    "Price <= 0": (df["Price"] <= 0).sum(),
    "Cancelled invoices": df["Invoice"].astype(str).str.startswith("C").sum(),
}
before_df = pd.DataFrame(before, index=["Before cleaning"]).T
before_df

,Before cleaning
Total rows,1067371
Missing Customer ID,243007
Missing Description,4382
Exact duplicate rows,34335
Quantity <= 0,22950
Price <= 0,6207
Cancelled invoices,19494


In [5]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


## Cleaning Step 1: Duplicates, Cancellations, Invalid Values, Non-Product Codes

In [8]:
# Make a copy so the original "df" stays untouched (useful for before/after comparisons)
df_clean = df.copy()

# Make sure Invoice and StockCode are treated as text, not numbers
df_clean["Invoice"] = df_clean["Invoice"].astype(str)
df_clean["StockCode"] = df_clean["StockCode"].astype(str)

print("Rows at the start:", len(df_clean))

# Remove rows that are exact copies of another row
df_clean = df_clean.drop_duplicates()

print("Rows after removing duplicates:", len(df_clean))

Rows at the start: 1067371
Rows after removing duplicates: 1033036


In [9]:
# Cancelled orders have an invoice number starting with the letter "C"
# This gives True for cancelled rows and False for the rest
is_cancelled = df_clean["Invoice"].str.startswith("C")

# Save the cancelled rows in their own table (we need them later for the cancellation rate)
cancellations = df_clean[is_cancelled]

# Keep only the rows that are NOT cancelled (the ~ symbol means "not")
df_clean = df_clean[~is_cancelled]

print("Cancelled rows saved separately:", len(cancellations))
print("Rows remaining:", len(df_clean))

Cancelled rows saved separately: 19104
Rows remaining: 1013932


In [10]:
# A real sale must have a quantity above 0 AND a price above 0
# The & symbol means "and"
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["Price"] > 0)]

print("Rows remaining:", len(df_clean))

Rows remaining: 1007913


In [11]:
# A real product code starts with 5 digits, like 85048 or 79323P
# This gives True for real products and False for everything else
is_product = df_clean["StockCode"].str.match(r"^\d{5}")

# Show the most common codes that are NOT real products
print(df_clean.loc[~is_product, "StockCode"].value_counts().head(10))

StockCode
POST            1851
DOT             1415
M                846
C2               267
ADJUST            36
BANK CHARGES      34
DCGS0058          30
gift_0001_20      26
gift_0001_30      24
DCGSSGIRL         23
Name: count, dtype: int64


In [12]:
# Keep only the real products
df_clean = df_clean[is_product]

print("Rows remaining:", len(df_clean))

Rows remaining: 1003214


## Cleaning Step 2: Missing Values, New Columns, Export

In [13]:
# Count the missing values left in each column
print(df_clean.isnull().sum())

Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    226637
Country             0
dtype: int64


**Decision:** About 22% of rows have no Customer ID. These are still real sales, so I keep them
for revenue analysis. I will exclude them only from customer-level analysis (RFM and cohorts),
because we can't tell which customer they belong to.

In [14]:
# Remove extra spaces and make descriptions upper case so the same product isn't spelled two ways
df_clean["Description"] = df_clean["Description"].str.strip().str.upper()
df_clean["Country"] = df_clean["Country"].str.strip()

In [15]:
# Revenue = how many items were bought x the price of each item
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["Price"]

print("Total revenue:", round(df_clean["Revenue"].sum(), 2))

Total revenue: 19642692.15


In [16]:
# Pull useful pieces out of the InvoiceDate column
df_clean["Year"] = df_clean["InvoiceDate"].dt.year
df_clean["Month"] = df_clean["InvoiceDate"].dt.month
df_clean["MonthName"] = df_clean["InvoiceDate"].dt.month_name()
df_clean["Weekday"] = df_clean["InvoiceDate"].dt.day_name()
df_clean["Hour"] = df_clean["InvoiceDate"].dt.hour

# Year and month together, like "2010-03", handy for monthly trend charts
df_clean["YearMonth"] = df_clean["InvoiceDate"].dt.to_period("M").astype(str)

df_clean.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,MonthName,Weekday,Hour,YearMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,12,December,Tuesday,7,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,Tuesday,7,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,Tuesday,7,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,12,December,Tuesday,7,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,12,December,Tuesday,7,2009-12


In [17]:
# Keep only the rows where we know who the customer is
df_customers = df_clean.dropna(subset=["Customer ID"]).copy()

# Customer ID was a decimal (13085.0), so turn it into a whole number (13085)
df_customers["Customer ID"] = df_customers["Customer ID"].astype(int)

print("Rows with a customer:", len(df_customers))
print("Unique customers:", df_customers["Customer ID"].nunique())

Rows with a customer: 776577
Unique customers: 5852


In [18]:
# Save to the data/processed folder (the ../ goes up one level out of notebooks)
df_clean.to_csv("../data/processed/clean_data.csv", index=False)
df_customers.to_csv("../data/processed/clean_customers.csv", index=False)
cancellations.to_csv("../data/processed/cancellations.csv", index=False)

print("Files saved!")

Files saved!
